# Stage 3 & 4 — Ranking, Match Score & Full Analysis
Combine sentiment + tag signals with Yelp metadata into a Match Score for any natural-language query.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import (
    build_restaurant_profiles, make_sample_dataset,
    preprocess_dataframe, get_tag_columns,
)
from ranking.match_score import parse_query, compute_match_scores, ablation_study
from evaluation.metrics import (
    plot_match_score_distribution, plot_score_breakdown,
)

sns.set_theme(style='whitegrid')
%matplotlib inline
os.makedirs('../data/figures', exist_ok=True)

In [ ]:
# Load restaurant profiles (build from scratch if not available)
try:
    profiles = pd.read_parquet('../data/restaurant_profiles.parquet')
    print(f'Loaded {len(profiles):,} restaurant profiles.')
except FileNotFoundError:
    print('Profiles not found — building from synthetic data.')
    df = preprocess_dataframe(make_sample_dataset(n=3000))
    profiles = build_restaurant_profiles(df)

profiles.head(3)

## Example Query 1: Vegan brunch spots with a quiet, aesthetic vibe

In [ ]:
QUERY_1 = 'Vegan brunch spots with a quiet, aesthetic vibe in San Francisco'
parsed_1 = parse_query(QUERY_1)
print('Parsed query:', parsed_1)

results_1 = compute_match_scores(profiles, parsed_1, top_k=20)
print(f'\nTop {len(results_1)} results:')
display_cols = ['name', 'city', 'match_score', 'score_sentiment', 'score_tag_match', 'score_stars']
display_cols = [c for c in display_cols if c in results_1.columns]
results_1[display_cols].head(10)

In [ ]:
plot_match_score_distribution(
    results_1.head(15), query=QUERY_1,
    save_path='../data/figures/match_scores_q1.png'
);

In [ ]:
# Score breakdown for top result
plot_score_breakdown(results_1.iloc[0], save_path='../data/figures/score_breakdown_top1.png');

## Example Query 2: Late-night trendy Korean BBQ

In [ ]:
QUERY_2 = 'Late night trendy Korean BBQ spot in Las Vegas'
parsed_2 = parse_query(QUERY_2)
results_2 = compute_match_scores(profiles, parsed_2, top_k=20)
plot_match_score_distribution(
    results_2.head(15), query=QUERY_2,
    save_path='../data/figures/match_scores_q2.png'
);

## Ablation Study — Contribution of each scoring component

In [ ]:
ablation_df = ablation_study(profiles, parsed_1)
print(ablation_df.pivot_table(index='rank', columns='ablation', values='match_score').round(2))

In [ ]:
# Weight sensitivity curves
weight_vals = np.linspace(0, 1, 11)
avg_scores = []
for w_sent in weight_vals:
    remaining = 1 - w_sent
    w = {
        'sentiment': w_sent,
        'tag_match': remaining * 0.4,
        'stars':     remaining * 0.3,
        'volume':    remaining * 0.15,
        'cuisine':   remaining * 0.15,
    }
    top = compute_match_scores(profiles, parsed_1, weights=w, top_k=20)
    avg_scores.append(top['match_score'].mean())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(weight_vals, avg_scores, marker='o', color='steelblue')
ax.set_xlabel('Sentiment weight')
ax.set_ylabel('Avg Match Score of Top-20')
ax.set_title('Weight Sensitivity: Sentiment Component')
plt.tight_layout()
plt.savefig('../data/figures/weight_sensitivity.png', dpi=150)
plt.show()

## Score Distribution Plot

In [ ]:
all_results = compute_match_scores(profiles, parsed_1, top_k=len(profiles))
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(all_results['match_score'], bins=40, color='coral', edgecolor='white')
ax.set_xlabel('Match Score')
ax.set_ylabel('# Restaurants')
ax.set_title('Distribution of Match Scores (all restaurants)')
plt.tight_layout()
plt.savefig('../data/figures/score_distribution.png', dpi=150)
plt.show()